# Cone-spawn curriculum inspector

Interactive (Plotly) visualisation of the **segment-spawn approach cone** and its **cosine curriculum schedules**, so you can see how hard the approach actually gets as training progresses.

Everything is imported live from [`lsy_drone_racing/envs/segment_spawn.py`](../envs/segment_spawn.py) — the config values and the schedule functions are the *same objects training uses*, so this notebook never drifts from the real curriculum.

**Mental model** (from the module docstring):
- A spawn for target gate `i` is drawn from a truncated cone (frustum) whose **tip sits at the gate centre** and whose **axis points along the gate's entry side** (`-n`, the negative traversal normal).
- The cone radius grows with distance from the gate: `r(s) = s * tan(theta)`, `theta in [0, kappa*theta_max]`, `s in [d_min, d_min + kappa*(d_max - d_min)]`.
  - Close spawns are forced near the gate axis (well aligned with the opening).
  - Far spawns may be strongly off-axis → the policy must actively steer to line up.
- Two cosine knobs on separate `tau` windows drive difficulty: **`kappa`** (cone size) ramps first, **`p_start`** (true-start mixture) ramps later, and **`v0`** (initial speed through the gate) ramps *down* early.

`tau = global_step / total_timesteps` is normalised training progress in `[0, 1]`.

> The geometry figures use a **Plotly frame-slider** (drag `tau`). These are self-contained client-side animations — no kernel callback — so they render reliably in VSCode.

In [7]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from lsy_drone_racing.envs.segment_spawn import (
    SegmentSpawnConfig,
    cosine_ramp,
    kappa_schedule,
    p_start_schedule,
    v0_schedule,
)

# Gate geometry (config comment: 0.72 m outer frame, 0.40 m opening; pass-check box is 0.45 m).
GATE_OPENING = 0.40   # inner opening width/height [m]
GATE_OUTER = 0.72     # outer frame width/height [m]
GATE_PASS_BOX = 0.45  # tolerance box used by gate_passed() in race_core.py
GATE_HALF = GATE_OPENING / 2.0  # 0.20 m

cfg = SegmentSpawnConfig()
TAU_GRID = np.round(np.arange(0.0, 1.0001, 0.05), 2)  # slider stops


def _f(x):
    """jax scalar/array -> numpy."""
    return np.asarray(x)


def _tau_slider(frames, prefix="tau = "):
    """A Plotly slider that animates between per-tau frames (redraw=True for 3D safety)."""
    steps = [
        dict(method="animate", label=f.name,
             args=[[f.name], dict(mode="immediate", frame=dict(duration=0, redraw=True),
                                  transition=dict(duration=0))])
        for f in frames
    ]
    return [dict(active=0, currentvalue={"prefix": prefix}, pad={"t": 40}, steps=steps)]


print("Loaded SegmentSpawnConfig + schedules from lsy_drone_racing.envs.segment_spawn")

Loaded SegmentSpawnConfig + schedules from lsy_drone_racing.envs.segment_spawn


## 1. The parameters

All the static knobs that define the cone and its curriculum, pulled straight from `SegmentSpawnConfig`.

In [8]:
_DESC = {
    "gate_offset": "exit-point offset of predecessor gate (defines segment length) [m]",
    "d_min": "min standoff from the gate — always leave runway [m]",
    "d_max_cap": "global cap on segment length [m]",
    "theta_max": "cone half-angle at kappa=1 [rad]",
    "margin": "required horizontal clearance to any obstacle [m]",
    "z_min": "spawn altitude floor [m]",
    "z_max": "spawn altitude ceiling [m]",
    "n_candidates": "rejection-sampling budget per env",
    "a0": "(a) cone-size window start [tau]",
    "a1": "(a) cone-size window end [tau]",
    "kappa_min": "cone-size fraction at tau<=a0",
    "b0": "(b) true-start window start [tau]",
    "b1": "(b) true-start window end [tau]",
    "p_start_min": "true-start probability floor",
    "p_start_max": "true-start probability at tau>=b1",
    "c0": "(c) initial-speed window start [tau]",
    "c1": "(c) initial-speed window end [tau]",
    "v0_max": "initial speed along gate normal at tau=0 [m/s]",
}
print(f"{'param':>14}  {'value':>8}   description")
print("-" * 78)
for k, d in _DESC.items():
    print(f"{k:>14}  {getattr(cfg, k):>8}   {d}")
print("-" * 78)
print(f"{'GATE_OPENING':>14}  {GATE_OPENING:>8}   inner gate opening (half = {GATE_HALF} m)")
print(f"{'theta_max[deg]':>14}  {np.degrees(cfg.theta_max):>8.1f}   cone half-angle at kappa=1")
print(f"{'tan(theta_max)':>14}  {np.tan(cfg.theta_max):>8.3f}   lateral spread per metre at kappa=1")

         param     value   description
------------------------------------------------------------------------------
   gate_offset       0.1   exit-point offset of predecessor gate (defines segment length) [m]
         d_min      0.25   min standoff from the gate — always leave runway [m]
     d_max_cap       1.5   global cap on segment length [m]
     theta_max       0.4   cone half-angle at kappa=1 [rad]
        margin       0.3   required horizontal clearance to any obstacle [m]
         z_min       0.2   spawn altitude floor [m]
         z_max       2.0   spawn altitude ceiling [m]
  n_candidates        12   rejection-sampling budget per env
            a0      0.05   (a) cone-size window start [tau]
            a1       0.7   (a) cone-size window end [tau]
     kappa_min       0.1   cone-size fraction at tau<=a0
            b0      0.25   (b) true-start window start [tau]
            b1      0.85   (b) true-start window end [tau]
   p_start_min      0.05   true-start probability

## 2. The cosine schedules vs `tau`

Three knobs, three windows. Read top-to-bottom as the training timeline:

| knob | window | from → to | effect |
|---|---|---|---|
| `kappa` (cone size) | `[a0, a1]` | `kappa_min` → `1` | **ramps first** → widens the spawn cone (discovery) |
| `p_start` (true-start mix) | `[b0, b1]` | `p_start_min` → `p_start_max` | **ramps later** → shifts toward the deployment start distribution |
| `v0` (initial speed) | `[c0, c1]` | `v0_max` → `0` | **ramps down early** → removes the through-gate momentum crutch |

The shaded band on each panel is that knob's active window.

In [9]:
taus = np.linspace(0.0, 1.0, 400)
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                    subplot_titles=("kappa  (cone size)", "p_start  (true-start prob.)",
                                    "v0  (initial speed) [m/s]"))
specs = [
    (1, _f(kappa_schedule(taus, cfg)), (cfg.a0, cfg.a1), "royalblue"),
    (2, _f(p_start_schedule(taus, cfg)), (cfg.b0, cfg.b1), "seagreen"),
    (3, _f(v0_schedule(taus, cfg)), (cfg.c0, cfg.c1), "firebrick"),
]
for row, y, (w0, w1), c in specs:
    fig.add_trace(go.Scatter(x=taus, y=y, mode="lines", line=dict(color=c, width=3),
                             showlegend=False), row=row, col=1)
    fig.add_vrect(x0=w0, x1=w1, fillcolor=c, opacity=0.12, line_width=0, row=row, col=1)
fig.update_xaxes(title_text="tau = global_step / total_timesteps", row=3, col=1)
fig.update_xaxes(range=[0, 1])  # tau only ever spans [0, 1]
fig.update_layout(height=720, title="Curriculum schedules", template="plotly_white")
fig.show()

## 3. Cone shape vs gate size — side view (cross-section)

The single most useful view: a slice through the cone axis. **Drag the `tau` slider.**

- **x-axis** = distance from the gate along the entry axis (`s`); the gate plane is at `x = 0` (right edge — the drone flies right→through the gate).
- **y-axis** = lateral offset from the gate axis.
- The **thick red bar at x=0** is the gate opening (±0.20 m); the thin dotted bar is the outer frame.
- The **shaded wedge** is where cone spawns can land: at distance `s` the lateral offset reaches up to `s * tan(kappa*theta_max)`, between `d_min` and `s_hi = d_min + kappa*(d_max - d_min)`.
- The marker labels the **max off-axis offset at the far end** as a multiple of the gate half-opening — the alignment the policy must correct.

Segment length is fixed at `d_max = d_max_cap` (the worst case); edit `D_MAX` below to inspect shorter segments.

In [10]:
D_MAX = cfg.d_max_cap  # segment length to visualise (<= cfg.d_max_cap)


def _side_traces(tau, d_max):
    kap = float(kappa_schedule(tau, cfg))
    th = kap * cfg.theta_max
    d_min = cfg.d_min
    s_hi = max(d_min + kap * (d_max - d_min), d_min + 1e-3)
    s = np.linspace(d_min, s_hi, 120)
    r = s * np.tan(th)
    far = s_hi * np.tan(th)
    xpoly = np.concatenate([s, s[::-1]])
    ypoly = np.concatenate([r, -r[::-1]])
    s_full = np.linspace(0.0, s_hi, 40)
    return [
        go.Scatter(x=xpoly, y=ypoly, fill="toself", mode="lines", line=dict(color="royalblue"),
                   fillcolor="rgba(65,105,225,0.25)", name="spawn region"),
        go.Scatter(x=s_full, y=s_full * np.tan(th), mode="lines",
                   line=dict(color="royalblue", dash="dot", width=1), showlegend=False),
        go.Scatter(x=s_full, y=-s_full * np.tan(th), mode="lines",
                   line=dict(color="royalblue", dash="dot", width=1), showlegend=False),
        go.Scatter(x=[0, 0], y=[-GATE_HALF, GATE_HALF], mode="lines",
                   line=dict(color="firebrick", width=8), name="gate opening (0.40 m)"),
        go.Scatter(x=[0, 0], y=[-GATE_OUTER / 2, GATE_OUTER / 2], mode="lines",
                   line=dict(color="firebrick", width=2, dash="dot"), name="gate frame (0.72 m)"),
        go.Scatter(x=[s_hi], y=[far], mode="markers+text",
                   text=[f"  {far:.2f} m  ({far / GATE_HALF:.1f}x gate half)"],
                   textposition="middle right", marker=dict(color="royalblue", size=8),
                   name="far-end max"),
    ]


frames = [go.Frame(data=_side_traces(t, D_MAX), name=f"{t:.2f}") for t in TAU_GRID]
fig = go.Figure(data=_side_traces(TAU_GRID[0], D_MAX), frames=frames)
fig.update_xaxes(autorange="reversed", title="distance from gate along entry axis  s  [m]",
                 zeroline=True)
fig.update_yaxes(scaleanchor="x", scaleratio=1, title="lateral offset  [m]")
fig.update_layout(height=500
                  , template="plotly_white", title=f"Approach cone (side view), d_max={D_MAX:.2f} m",
                  sliders=_tau_slider(frames), legend=dict(orientation="h", y=1.08))
fig.show()

## 4. Cone vs gate — 3D with sampled spawns

The same cone in 3D, with actual sampled spawn points (same geometry as `_sample_cone_spawn`, minus the obstacle/altitude rejection). The gate sits in the y–z plane at the origin with its normal along +x; the drone flies in the +x direction through it. Spawns populate the entry side (−x). **Drag `tau`; rotate with the mouse.**

In [11]:
N_POINTS = 500
SEED = 0


def _square3d(half, color, width, name):
    return go.Scatter3d(x=[0, 0, 0, 0, 0], y=[-half, half, half, -half, -half],
                        z=[-half, -half, half, half, -half], mode="lines",
                        line=dict(color=color, width=width), name=name)


def _cone3d_traces(tau, d_max, n, seed):
    kap = float(kappa_schedule(tau, cfg))
    d_min = cfg.d_min
    s_hi = max(d_min + kap * (d_max - d_min), d_min + 1e-3)
    rng = np.random.default_rng(seed)
    s = rng.uniform(d_min, s_hi, n)
    theta = rng.uniform(0.0, kap * cfg.theta_max, n)
    phi = rng.uniform(0.0, 2 * np.pi, n)
    radial = s * np.tan(theta)
    return [
        go.Scatter3d(x=-s, y=radial * np.cos(phi), z=radial * np.sin(phi), mode="markers",
                     marker=dict(size=2, color="royalblue", opacity=0.5), name="sampled spawns"),
        _square3d(GATE_HALF, "firebrick", 6, "gate opening"),
        _square3d(GATE_OUTER / 2, "firebrick", 2, "gate frame"),
    ]


frames = [go.Frame(data=_cone3d_traces(t, cfg.d_max_cap, N_POINTS, SEED), name=f"{t:.2f}")
          for t in TAU_GRID]
fig = go.Figure(data=_cone3d_traces(TAU_GRID[0], cfg.d_max_cap, N_POINTS, SEED), frames=frames)
lim = cfg.d_max_cap * np.tan(cfg.theta_max) * 1.05
fig.update_layout(
    height=680, template="plotly_white", title="Approach cone in 3D (drag tau, rotate with mouse)",
    sliders=_tau_slider(frames),
    scene=dict(xaxis_title="x along axis [m]", yaxis_title="y [m]", zaxis_title="z [m]",
               yaxis=dict(range=[-lim, lim]), zaxis=dict(range=[-lim, lim]),
               xaxis=dict(range=[-cfg.d_max_cap * 1.05, 0.4]), aspectmode="data"),
)
fig.show()

## 5. How hard does the approach actually get?

Difficulty proxies as a function of `tau` (for segment length `d_max = d_max_cap`):

1. **Max off-axis offset at the far end** `= s_hi * tan(kappa*theta_max)`, in units of the gate half-opening. `1.0` = a far spawn can sit exactly at the edge of the opening; `>1` = fully outside the opening cone, so the policy must steer back in.
2. **Max standoff** `s_hi` — how far back the drone can start (runway to manage).

`v0` (momentum crutch) is shown because it offsets difficulty: while `v0 > 0` the drone is *launched* through the gate; once `v0 → 0` the wide cone is fully on the policy. `p_start` shows when the deployment start distribution takes over.

In [12]:
d_max = cfg.d_max_cap
taus = np.linspace(0.0, 1.0, 400)
kap = _f(kappa_schedule(taus, cfg))
s_hi = cfg.d_min + kap * (d_max - cfg.d_min)
max_lat = s_hi * np.tan(kap * cfg.theta_max)
v0 = _f(v0_schedule(taus, cfg))
p_start = _f(p_start_schedule(taus, cfg))

fig = make_subplots(rows=2, cols=2, vertical_spacing=0.13, horizontal_spacing=0.10,
                    subplot_titles=("max off-axis / gate half-opening",
                                    "v0 initial speed (momentum crutch) [m/s]",
                                    "max standoff s_hi (runway) [m]",
                                    "p_start (frac. from true race start)"))
fig.add_trace(go.Scatter(x=taus, y=max_lat / GATE_HALF, line=dict(color="royalblue", width=3),
                         showlegend=False), row=1, col=1)
fig.add_hline(y=1.0, line=dict(color="firebrick", dash="dot"), row=1, col=1)
fig.add_trace(go.Scatter(x=taus, y=v0, line=dict(color="firebrick", width=3), showlegend=False),
              row=1, col=2)
fig.add_trace(go.Scatter(x=taus, y=s_hi, line=dict(color="purple", width=3), showlegend=False),
              row=2, col=1)
fig.add_trace(go.Scatter(x=taus, y=p_start, line=dict(color="seagreen", width=3), showlegend=False),
              row=2, col=2)
fig.update_xaxes(title_text="tau", row=2, col=1)
fig.update_xaxes(title_text="tau", row=2, col=2)
fig.update_yaxes(title_text="x gate half (0.20 m)", row=1, col=1)
fig.update_xaxes(range=[0, 1])  # tau only ever spans [0, 1]
fig.update_layout(height=680, template="plotly_white",
                  title=f"Approach difficulty vs training progress (d_max = {d_max:.2f} m)")
fig.show()